In [1]:
# Install required packages (run once)
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Module 4: Procedural Memory

Modules 2–3 taught the agent **what to remember** — events and facts. But neither answers **how** to approach a task. That knowledge lives in the agent's instructions — and right now it's an unversioned Python string.

This module treats instructions as **procedural memory**: learned behaviors stored in version-controlled files, refined through iteration, and loaded on demand via the Skills pattern.

## What You'll Learn
1. Why instructions are a **third kind of memory** (not just configuration)
2. The **feedback loop**: Run → Observe → Refine → Verify
3. **Task-specific procedures** via SKILL.md files + `SkillsProvider`
4. **When to stop refining** — stopping signals and anti-patterns


---
## Setup

> **Prerequisites**:
> - Modules 1–3 completed
> - Azure AI Foundry project endpoint in your `.env` file
> - Git installed (for version tracking — optional but recommended)
>
> This module runs entirely with Azure AI Foundry — no additional infrastructure needed.
> Memory tools use in-memory stores so you can focus on the procedural memory concept.

In [2]:
import os
import json
import asyncio
import difflib
import nest_asyncio
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv("../../.env", override=True)

PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ.get("FOUNDRY_MODEL", "gpt-4o")
TENANT_ID = os.environ.get("AZURE_TENANT_ID", "16b3c013-d300-468d-ac64-7eda0820b6d3")

print(f"Project:  {PROJECT_ENDPOINT}")
print(f"Model:    {MODEL_DEPLOYMENT}")


def load_instructions(path: str) -> str:
    """Load agent instructions from a markdown file."""
    return Path(path).read_text(encoding="utf-8")


def show_diff(old_path: str, new_path: str) -> None:
    """Show a unified diff between two instruction files."""
    old_lines = Path(old_path).read_text(encoding="utf-8").splitlines(keepends=True)
    new_lines = Path(new_path).read_text(encoding="utf-8").splitlines(keepends=True)
    diff = difflib.unified_diff(old_lines, new_lines, fromfile=old_path, tofile=new_path)
    output = "".join(diff)
    if output:
        print(output)
    else:
        print("(no differences)")

print("\nHelpers defined: load_instructions(), show_diff()")

Project:  https://npmsfoundrysc.services.ai.azure.com/api/projects/proj-default
Model:    gpt-5.4-mini

Helpers defined: load_instructions(), show_diff()


---
## The Problem: Memory Without Procedure

Our travel assistant has two memory systems — but no workflow for using them:

| Memory Type | What It Stores | Example |
|---|---|---|
| **Episodic** | Events, experiences | "Sarah's NYC trip cost $1,760" |
| **Semantic** | Facts, preferences | "Sarah prefers Delta" |
| **???** | How to approach tasks | How to handle a booking request |

The missing piece is **procedural memory** — the knowledge of *how to do things*. Today that knowledge is buried in a Python string: unversioned, unreviewable, impossible to diff.

The fix: **treat instructions as files** — tracked by git, reviewed in PRs, loaded from the filesystem.

| **Procedural memory** (how to act) | **NOT procedural** (use other memory) |
|---|---|
| Booking workflow steps | "Sarah prefers Delta" (semantic) |
| Budget verification procedures | "NYC trip cost $1,760" (episodic) |
| When-to-use-which-memory rules | "Book me a flight Tuesday" (request) |


In [ ]:
# ─── Boilerplate from Modules 2–3: memory tools + preloaded data ───
# (Same episodic + semantic tools — focus is on PROCEDURAL memory this module)

from azure.identity import AzureCliCredential
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from agent_framework import tool

credential = AzureCliCredential()

# ─── Episodic memory (in-memory) ───

_episodic_store = []

@tool
async def remember_event(
    user_id: str,
    event_type: str,
    description: str,
    details: str = "",
) -> str:
    """Store a significant event in the user's episodic memory.

    Call this when:
    - A trip is completed or booked
    - The user shares feedback about a past experience
    - Something notable happens worth remembering for the future

    Args:
        user_id: Employee ID (e.g. "E001")
        event_type: Category — "trip", "booking", "feedback", "preference"
        description: Brief summary of what happened
        details: Optional JSON string with structured data
    """
    event = {
        "id": f"{user_id}-{uuid4().hex[:8]}",
        "user_id": user_id,
        "event_type": event_type,
        "description": description,
        "details": json.loads(details) if details else {},
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    _episodic_store.append(event)
    return f"Remembered event: {description}"


@tool
async def recall_events(
    user_id: str,
    query: str = "",
    event_type: str = "",
    limit: int = 5,
) -> str:
    """Search the user's episodic memory for past events.

    Call this when:
    - Planning a trip and want to check past experiences
    - The user asks about their travel history
    - Need context from previous trips for recommendations

    Args:
        user_id: Employee ID (e.g. "E001")
        query: Optional keyword to search in descriptions
        event_type: Optional filter — "trip", "booking", "feedback"
        limit: Maximum number of events to return
    """
    results = [e for e in _episodic_store if e["user_id"] == user_id]
    if event_type:
        results = [e for e in results if e["event_type"] == event_type]
    if query:
        results = [e for e in results if query.lower() in e["description"].lower()]
    results = sorted(results, key=lambda e: e["timestamp"], reverse=True)[:limit]
    return json.dumps(results, indent=2) if results else "No matching events found."


# ─── Semantic memory (in-memory) ───

class OntologyManager:
    RELATIONSHIP_MAP = {
        "prefers": "PREFERS", "likes": "PREFERS", "prefer": "PREFERS",
        "loves": "PREFERS", "always": "PREFERS", "favorite": "PREFERS",
        "dislikes": "DISLIKES", "hates": "DISLIKES", "avoids": "DISLIKES",
        "never": "DISLIKES", "dont_like": "DISLIKES",
        "requires": "REQUIRES", "needs": "REQUIRES", "must_have": "REQUIRES",
        "works_at": "WORKS_AT", "employed_at": "WORKS_AT",
        "member_of": "MEMBER_OF", "belongs_to": "MEMBER_OF",
    }
    ENTITY_TYPE_MAP = {
        "airline": "AIRLINE", "carrier": "AIRLINE",
        "hotel": "HOTEL", "hotel_chain": "HOTEL",
        "seat": "SEAT_TYPE", "seat_type": "SEAT_TYPE",
        "dietary": "DIETARY", "diet": "DIETARY", "food": "DIETARY",
        "department": "DEPARTMENT", "team": "DEPARTMENT",
        "accessibility": "ACCESSIBILITY",
    }
    def canonicalize_relationship(self, raw):
        return self.RELATIONSHIP_MAP.get(raw.lower().strip().replace(" ", "_"), raw.upper())
    def canonicalize_entity_type(self, raw):
        return self.ENTITY_TYPE_MAP.get(raw.lower().strip().replace(" ", "_"), raw.upper())


class SmartKnowledgeGraph:
    def __init__(self):
        self.graph = {}
        self.ontology = OntologyManager()

    def store_triplet(self, user_id, subject, relationship, obj, object_type=""):
        rel = self.ontology.canonicalize_relationship(relationship)
        otype = self.ontology.canonicalize_entity_type(object_type) if object_type else ""
        if user_id not in self.graph:
            self.graph[user_id] = []
        if otype:
            self.graph[user_id] = [
                t for t in self.graph[user_id]
                if not (t["relationship"] == rel and t["object_type"] == otype)
            ]
        triplet = {
            "subject": subject, "relationship": rel, "object": obj,
            "object_type": otype, "timestamp": datetime.now(timezone.utc).isoformat(),
        }
        self.graph[user_id].append(triplet)
        return triplet

    def query(self, user_id, relationship=None, object_type=None):
        results = self.graph.get(user_id, [])
        if relationship:
            results = [t for t in results if t["relationship"] == relationship]
        if object_type:
            results = [t for t in results if t["object_type"] == object_type]
        return results

    def all_facts(self, user_id):
        return self.graph.get(user_id, [])


_knowledge_graph = SmartKnowledgeGraph()


EXTRACTION_PROMPT = """Extract knowledge triplets from the user's statement.
Return a JSON array of objects with these fields:
- "relationship": how the subject relates to the object (e.g. "prefers", "requires", "dislikes")
- "object": the entity (e.g. "Delta", "aisle", "vegetarian")
- "object_type": category (e.g. "airline", "hotel", "seat_type", "dietary", "accessibility")

Rules:
- Only extract PERSISTENT facts (preferences, requirements) — not one-time plans
- "I need to fly Tuesday" is NOT a preference — skip it
- "I always fly United" IS a preference — extract it
- Return ONLY the JSON array, no other text

Examples:
Input: "I prefer Delta and I'm vegetarian"
Output: [{"relationship": "prefers", "object": "Delta", "object_type": "airline"}, {"relationship": "requires", "object": "vegetarian", "object_type": "dietary"}]

Input: "Book me a flight to NYC next week"
Output: []
"""


async def extract_knowledge_triplets(statement):
    extractor = Agent(
        client=FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        ),
        name="Extractor",
        instructions=EXTRACTION_PROMPT,
    )
    response = await extractor.run(statement)
    text = response.text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text[3:]
        text = text.rsplit("```", 1)[0].strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return []


@tool
async def learn_about_user(user_id: str, statement: str) -> str:
    """Extract and store facts/preferences from a user's statement in semantic memory.

    Args:
        user_id: Employee ID (e.g. "E001")
        statement: Natural language containing facts to remember
    """
    triplets = await extract_knowledge_triplets(statement)
    if not triplets:
        return "No persistent facts found in that statement."

    ontology = OntologyManager()
    stored = []
    for t in triplets:
        _knowledge_graph.store_triplet(
            user_id, user_id, t["relationship"], t["object"], t.get("object_type", ""),
        )
        rel = ontology.canonicalize_relationship(t["relationship"])
        otype = ontology.canonicalize_entity_type(t.get("object_type", ""))
        stored.append(f"({user_id}, {rel}, {t['object']}/{otype})")
    return f"Learned {len(stored)} facts: " + "; ".join(stored)


@tool
async def query_user_knowledge(
    user_id: str,
    relationship: str = "",
    entity_type: str = "",
) -> str:
    """Query the user's semantic memory for stored facts and preferences.

    Args:
        user_id: Employee ID (e.g. "E001")
        relationship: Optional filter — "PREFERS", "REQUIRES", "DISLIKES"
        entity_type: Optional filter — "AIRLINE", "HOTEL", "SEAT_TYPE", "DIETARY"
    """
    ontology = OntologyManager()
    rel = ontology.canonicalize_relationship(relationship) if relationship else None
    etype = ontology.canonicalize_entity_type(entity_type) if entity_type else None
    facts = _knowledge_graph.query(user_id, relationship=rel, object_type=etype)
    if not facts:
        return "No matching facts found."
    lines = [f"({f['subject']}, {f['relationship']}, {f['object']}/{f['object_type']})" for f in facts]
    return "\n".join(lines)


# ─── Preload data ───

_episodic_store.extend([
    {"id": "E001-trip1", "user_id": "E001", "event_type": "trip",
     "description": "New York trip for tech conference (Oct 15-18, 2025). Marriott Times Square, rated 5/5.",
     "details": {"destination": "New York", "hotel": "Marriott Times Square", "rating": 5, "total_cost": 1760},
     "timestamp": "2025-10-15T00:00:00"},
    {"id": "E001-trip2", "user_id": "E001", "event_type": "trip",
     "description": "London trip for client meeting (Nov 20-25, 2025). Marriott Park Lane, rated 4/5.",
     "details": {"destination": "London", "hotel": "Marriott Park Lane", "rating": 4, "total_cost": 3900},
     "timestamp": "2025-11-20T00:00:00"},
])

_knowledge_graph.store_triplet("E001", "E001", "prefers", "Delta", "airline")
_knowledge_graph.store_triplet("E001", "E001", "prefers", "aisle", "seat_type")
_knowledge_graph.store_triplet("E001", "E001", "prefers", "Marriott", "hotel")
_knowledge_graph.store_triplet("E001", "E001", "requires", "vegetarian", "dietary")

print("Tools defined:")
print("  Episodic:  remember_event, recall_events")
print("  Semantic:  learn_about_user, query_user_knowledge")
print()
print("Preloaded data:")
print(f"  {len(_episodic_store)} episodic events")
print(f"  {len(_knowledge_graph.all_facts('E001'))} semantic facts")

Tools defined:
  Episodic:  remember_event, recall_events
  Semantic:  learn_about_user, query_user_knowledge

Preloaded data:
  2 episodic events
  4 semantic facts


In [ ]:
# Create the prompts directory
os.makedirs("prompts", exist_ok=True)

V1_INSTRUCTIONS = """You are a corporate travel assistant for Contoso Corp.
Help employees book business travel including flights and hotels.

## Episodic Memory — Past Events
You have access to the employee's episodic memory — their past trips and experiences.
When planning new trips:
1. Use recall_events to check if they've been to this destination before
2. Reference past experiences to make better recommendations
3. Use remember_event to store significant outcomes after bookings

## Semantic Memory — Facts & Preferences
You also have access to semantic memory — the user's facts, preferences, and relationships.

WHEN TO LEARN (call learn_about_user):
- User states a preference: "I prefer...", "I like...", "I always..."
- User shares a persistent fact: "I'm vegetarian", "I need wheelchair access"
- User corrects a previous preference: "Actually, I've switched to..."

WHEN TO QUERY (call query_user_knowledge):
- Before recommending flights or hotels → check known preferences first
- User asks about their own profile: "What do you know about me?"
- Making any personalized decision

WHEN TO SKIP:
- One-time situational info: "Book me a flight for Tuesday" (date, not a preference)
- Events that happened (use episodic memory for those, not semantic)
- Small talk, greetings, confirmations

WHEN FACTS CONFLICT:
- If the user explicitly corrects a preference, update immediately via learn_about_user
- If you detect a contradiction with stored knowledge, ASK the user before overwriting
- Newer explicit statements from the user override older ones

The current user is Sarah Chen (employee ID: E001, based in Seattle, USA).
Be concise and helpful.
"""

Path("prompts/v1.md").write_text(V1_INSTRUCTIONS, encoding="utf-8")
print("Created: prompts/v1.md")
print(f"Size: {len(V1_INSTRUCTIONS)} characters, {len(V1_INSTRUCTIONS.splitlines())} lines")

Created: prompts/v1.md
Size: 1644 characters, 35 lines


In [ ]:
async def demo_v1():
    instructions = load_instructions("prompts/v1.md")
    print(f"Loaded instructions from: prompts/v1.md ({len(instructions)} chars)\n")

    agent = Agent(
        client=FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        ),
        name="TravelAssistant",
        instructions=instructions,
        tools=[remember_event, recall_events, learn_about_user, query_user_knowledge],
    )
    turns = [
        "Hi, I'm Sarah Chen (E001). Can you help me plan a trip to Chicago?",
        "Book me a $3,000 business class flight to Chicago.",
    ]
    for user_msg in turns:
        print(f"User: {user_msg}")
        response = await agent.run(user_msg)
        print(f"Agent: {response.text}\n")

print("=== V1: Module 3 instructions loaded from file ===\n")
asyncio.run(demo_v1())

=== V1: Module 3 instructions loaded from file ===

Loaded instructions from: prompts/v1.md (1644 chars)

User: Hi, I'm Sarah Chen (E001). Can you help me plan a trip to Chicago?
Agent: Absolutely, Sarah — I can help with Chicago travel.

I found these preferences on your profile:
- Prefers Delta
- Prefers aisle seats
- Prefers Marriott hotels
- Requires vegetarian options

I didn’t find any past Chicago trip history in your memory.

To get started, please send:
1. Your travel dates
2. Departure city/airport
3. Number of travelers
4. Whether you want flight, hotel, or both
5. Any budget or schedule constraints

If you want, I can recommend options that match your preferences right away.

User: Book me a $3,000 business class flight to Chicago.
Agent: Sure — I can help with that.

To book the right flight, I need:
1. Your departure city or airport
2. Your travel dates
3. Whether you want nonstop only
4. Any airline or schedule preferences

You mentioned a $3,000 business class budget to

### V1 Has No Procedure

The agent checked preferences and recalled past trips — but it didn't question the **$3,000 business class** request. Sarah is a Senior Engineer with an $800 flight limit.

The instructions tell the agent *when to use memory*, but not *how to use it in a booking workflow*. That's the gap: no step-by-step procedure means no budget enforcement.

The fix isn't more tools — it's teaching the agent a **procedure**. Let's add one.


In [ ]:
# The feedback loop: Run → Observe weakness → Refine → Verify
# V1 weakness: no budget check → V2 adds a Booking Procedure section

V2_INSTRUCTIONS = """You are a corporate travel assistant for Contoso Corp.
Help employees book business travel including flights and hotels.

## Episodic Memory — Past Events
You have access to the employee's episodic memory — their past trips and experiences.
When planning new trips:
1. Use recall_events to check if they've been to this destination before
2. Reference past experiences to make better recommendations
3. Use remember_event to store significant outcomes after bookings

## Semantic Memory — Facts & Preferences
You also have access to semantic memory — the user's facts, preferences, and relationships.

WHEN TO LEARN (call learn_about_user):
- User states a preference: "I prefer...", "I like...", "I always..."
- User shares a persistent fact: "I'm vegetarian", "I need wheelchair access"
- User corrects a previous preference: "Actually, I've switched to..."

WHEN TO QUERY (call query_user_knowledge):
- Before recommending flights or hotels → check known preferences first
- User asks about their own profile: "What do you know about me?"
- Making any personalized decision

WHEN TO SKIP:
- One-time situational info: "Book me a flight for Tuesday" (date, not a preference)
- Events that happened (use episodic memory for those, not semantic)
- Small talk, greetings, confirmations

WHEN FACTS CONFLICT:
- If the user explicitly corrects a preference, update immediately via learn_about_user
- If you detect a contradiction with stored knowledge, ASK the user before overwriting
- Newer explicit statements from the user override older ones

## Booking Procedure
When the user requests a booking, follow these steps IN ORDER:

1. **Check preferences** (semantic memory): query_user_knowledge for airline, hotel, seat preferences
2. **Check history** (episodic memory): recall_events for past trips to the same destination
3. **Verify budget**: Senior employees have a $800 flight limit, economy/economy-plus only.
   - If the request exceeds the limit, explain the policy and suggest alternatives
   - Do NOT silently book over-budget options
4. **Explain your reasoning**: Tell the user WHY you're recommending what you recommend
   - "Based on your preference for Delta and your past positive experience at Marriott..."
   - "I notice this exceeds the $800 Senior-level flight budget, so here are alternatives..."
5. **Confirm before booking**: Always summarize the plan and ask for confirmation

The current user is Sarah Chen (employee ID: E001, level: Senior, based in Seattle, USA).
Be concise and helpful.
"""

Path("prompts/v2.md").write_text(V2_INSTRUCTIONS, encoding="utf-8")
print("Created: prompts/v2.md")
print(f"Size: {len(V2_INSTRUCTIONS)} characters, {len(V2_INSTRUCTIONS.splitlines())} lines")

Created: prompts/v2.md
Size: 2516 characters, 48 lines


In [ ]:
print("=== What changed: V1 → V2 ===\n")
show_diff("prompts/v1.md", "prompts/v2.md")

=== What changed: V1 → V2 ===

--- prompts/v1.md
+++ prompts/v2.md
@@ -31,5 +31,18 @@
 - If you detect a contradiction with stored knowledge, ASK the user before overwriting
 - Newer explicit statements from the user override older ones
 
-The current user is Sarah Chen (employee ID: E001, based in Seattle, USA).
+## Booking Procedure
+When the user requests a booking, follow these steps IN ORDER:
+
+1. **Check preferences** (semantic memory): query_user_knowledge for airline, hotel, seat preferences
+2. **Check history** (episodic memory): recall_events for past trips to the same destination
+3. **Verify budget**: Senior employees have a $800 flight limit, economy/economy-plus only.
+   - If the request exceeds the limit, explain the policy and suggest alternatives
+   - Do NOT silently book over-budget options
+4. **Explain your reasoning**: Tell the user WHY you're recommending what you recommend
+   - "Based on your preference for Delta and your past positive experience at Marriott

In [ ]:
async def demo_v2():
    instructions = load_instructions("prompts/v2.md")
    print(f"Loaded instructions from: prompts/v2.md ({len(instructions)} chars)\n")

    agent = Agent(
        client=FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        ),
        name="TravelAssistant",
        instructions=instructions,
        tools=[remember_event, recall_events, learn_about_user, query_user_knowledge],
    )
    # Same conversation as V1 — same user, same requests
    turns = [
        "Hi, I'm Sarah Chen (E001). Can you help me plan a trip to Chicago?",
        "Book me a $3,000 business class flight to Chicago.",
    ]
    for user_msg in turns:
        print(f"User: {user_msg}")
        response = await agent.run(user_msg)
        print(f"Agent: {response.text}\n")

print("=== V2: + Booking Procedure ===\n")
asyncio.run(demo_v2())

=== V2: + Booking Procedure ===

Loaded instructions from: prompts/v2.md (2516 chars)

User: Hi, I'm Sarah Chen (E001). Can you help me plan a trip to Chicago?
Agent: Absolutely — I can help plan your Chicago trip.

What I know so far:
- Preferred airline: Delta
- Preferred seat: aisle
- Preferred hotel: Marriott
- Dietary need: vegetarian

I checked your past trips and don’t see any prior Chicago trip in memory.

To plan next, send me:
- travel dates
- departure city/airport if different from Seattle
- whether you want flight + hotel or just one of them
- any budget or schedule constraints

I’ll use your preferences to suggest the best options.

User: Book me a $3,000 business class flight to Chicago.
Agent: I can’t book that as requested.

You’re a Senior employee, and Contoso’s flight policy is economy/economy-plus only with an $800 flight limit. A $3,000 business class flight to Chicago exceeds policy.

What I do know:
- Airline preference: Delta
- Seat preference: aisle
- No past 

### Same Tools, Different Behavior

| | V1 | V2 |
|---|---|---|
| **$3,000 request** | May book without questioning | Flags the $800 Senior limit |
| **Preferences** | Uses them if it remembers | Checks systematically (step 1) |
| **Past trips** | May or may not check | Checks before recommending (step 2) |
| **Reasoning** | Often silent | Explains why (step 4) |

The tools didn't change. What changed was the **procedure** — the step-by-step playbook for handling a booking. This is the core insight: **instructions are procedural memory**.

But V2 has one procedure for all trips. Real workflows are context-dependent: international trips need visa checks, insurance, and advance booking rules. The agent needs task-specific procedures that activate based on context.


---
## Context-Dependent Procedures via Skills

A general booking procedure isn't enough. Domestic trips and international trips need different checklists:

| Domestic | International |
|---|---|
| Check preferences | Check preferences |
| Check past trips | Check past trips + **visa history** |
| Verify budget | Verify budget |
| | **Travel insurance requirement** |
| | **14-day advance booking** |
| Confirm | Confirm |

The procedure *orchestrates* the other memory systems — deciding which memories to consult, in what order, and what to do with the results.

### Design Requirements → Skills Pattern

Procedures should be: **versionable** (git), **human-reviewed** (PRs), **task-specific** (load on demand), **extensible** (add a folder, no code change), and **bundle supporting data** (budget JSON, visa checklists).

The Agent Framework's **`SkillsProvider`** satisfies all five:
- Each procedure lives in a `SKILL.md` file with YAML frontmatter
- Supporting data lives alongside as resources
- Progressive disclosure: advertise (~50 tokens) → load (full procedure) → resources (data)
- New procedure = new folder — no code change, no redeployment


In [ ]:
import shutil

# Create the skills directory structure
os.makedirs("skills/domestic-booking", exist_ok=True)
os.makedirs("skills/international-booking", exist_ok=True)

# ─── Domestic booking skill ───

DOMESTIC_SKILL = """---
name: domestic-booking
description: "Step-by-step procedure for booking domestic (same-country) business travel. Use when employee requests a flight, hotel, or trip within their home country."
---

# Domestic Booking Procedure

Follow these steps when booking travel within the same country:

1. **Check preferences**: Query semantic memory for airline, hotel, and seat preferences
2. **Check history**: Recall past trips to the same city. If the user has been there before, reference their experience (hotel rating, what worked/didn't)
3. **Verify budget**: Use `read_skill_resource("domestic-booking", "budget-limits")` to get budget limits by employee level. Check the booking amount against the limit.
4. **Recommend options**: Use preferences and history to suggest specific flights/hotels
5. **Explain reasoning**: Tell the user WHY you're recommending each option
6. **Confirm**: Summarize the complete plan and get user approval before booking
"""

# ─── International booking skill ───

INTERNATIONAL_SKILL = """---
name: international-booking
description: "Step-by-step procedure for booking international business travel. Use when employee requests travel to a different country. Includes visa, insurance, and advance booking requirements."
---

# International Booking Procedure

Follow these steps when booking travel to a different country:

1. **Check preferences**: Query semantic memory for airline, hotel, and seat preferences
2. **Check history**: Recall past international trips. Check for visa-related events or issues
3. **Verify budget**: Use `read_skill_resource("international-booking", "budget-limits")` to get budget limits. International trips often exceed domestic limits — flag and escalate if needed
4. **Travel insurance**: International trips REQUIRE travel insurance per company policy. Remind the user and include in the plan
5. **Advance booking**: International trips require 14-day advance booking per company policy. Check the requested dates and warn if too close
6. **Visa check**: Ask if the user has a valid visa for the destination country. Use `read_skill_resource("international-booking", "visa-checklist")` for requirements. Check episodic memory for past visits (which implies prior visa)
7. **Recommend options**: Prefer direct flights for international routes when within budget
8. **Explain reasoning**: Include policy requirements in your explanation
9. **Confirm**: Summarize the complete plan including insurance and visa status
"""

# ─── Shared resources ───

BUDGET_LIMITS = json.dumps({
    "by_level": {
        "Junior":   {"max_flight_cost": 500,   "max_hotel_per_night": 200,  "allowed_flight_class": ["economy"]},
        "Senior":   {"max_flight_cost": 800,   "max_hotel_per_night": 300,  "allowed_flight_class": ["economy", "economy_plus"]},
        "Director": {"max_flight_cost": 1500,  "max_hotel_per_night": 400,  "allowed_flight_class": ["economy", "economy_plus", "business"]},
        "VP":       {"max_flight_cost": 3000,  "max_hotel_per_night": 500,  "allowed_flight_class": ["economy", "economy_plus", "business"]},
        "C-Suite":  {"max_flight_cost": 10000, "max_hotel_per_night": 1000, "allowed_flight_class": ["economy", "economy_plus", "business", "first"]},
    },
    "general": {
        "max_trip_duration_days": 14,
        "requires_business_justification": True,
    },
}, indent=2)

VISA_CHECKLIST = """# Visa Requirements Checklist

Before confirming any international booking:

1. **Ask the traveler** if they have a valid visa for the destination country
2. **Check episodic memory** — past trips to the same country imply prior visa (but check expiry)
3. **Visa processing time** — if a new visa is needed, factor in processing time before the trip
4. **Passport validity** — many countries require 6+ months remaining on passport
5. **Transit visas** — check if connecting flights transit through countries requiring a visa

If visa status is unclear, flag it as a blocker and do NOT confirm the booking.
"""

# Write skill files
Path("skills/domestic-booking/SKILL.md").write_text(DOMESTIC_SKILL, encoding="utf-8")
Path("skills/domestic-booking/budget-limits.json").write_text(BUDGET_LIMITS, encoding="utf-8")

Path("skills/international-booking/SKILL.md").write_text(INTERNATIONAL_SKILL, encoding="utf-8")
Path("skills/international-booking/budget-limits.json").write_text(BUDGET_LIMITS, encoding="utf-8")
Path("skills/international-booking/visa-checklist.md").write_text(VISA_CHECKLIST, encoding="utf-8")

print("Created skill files:")
for p in sorted(Path("skills").rglob("*")):
    if p.is_file():
        print(f"  {p}  ({len(p.read_text(encoding='utf-8').splitlines())} lines)")

Created skill files:
  skills\domestic-booking\budget-limits.json  (51 lines)
  skills\domestic-booking\SKILL.md  (15 lines)
  skills\international-booking\budget-limits.json  (51 lines)
  skills\international-booking\SKILL.md  (18 lines)
  skills\international-booking\visa-checklist.md  (11 lines)


In [ ]:
from agent_framework import SkillsProvider

# Create a SkillsProvider that discovers SKILL.md files automatically.
# This replaces our manual load_procedure() tool with the framework-native pattern.
procedures = SkillsProvider(skill_paths=["skills"])

# Show what was discovered
print("SkillsProvider discovered:\n")
for skill in procedures._skills.values():
    desc = skill.description[:80] + "..." if len(skill.description) > 80 else skill.description
    print(f"  📋 {skill.name}: {desc}")
    if skill.resources:
        for rname in skill.resources:
            print(f"     └── resource: {rname}")
    print()

print("Tools exposed:")
for t in procedures._tools:
    print(f"  🔧 {t.name}")


# ─── V3: Instructions referencing SkillsProvider ───

V3_INSTRUCTIONS = """You are a corporate travel assistant for Contoso Corp.
Help employees book business travel including flights and hotels.

## Episodic Memory — Past Events
You have access to the employee's episodic memory — their past trips and experiences.
When planning new trips:
1. Use recall_events to check if they've been to this destination before
2. Reference past experiences to make better recommendations
3. Use remember_event to store significant outcomes after bookings

## Semantic Memory — Facts & Preferences
You also have access to semantic memory — the user's facts, preferences, and relationships.

WHEN TO LEARN (call learn_about_user):
- User states a preference: "I prefer...", "I like...", "I always..."
- User shares a persistent fact: "I'm vegetarian", "I need wheelchair access"
- User corrects a previous preference: "Actually, I've switched to..."

WHEN TO QUERY (call query_user_knowledge):
- Before recommending flights or hotels → check known preferences first
- User asks about their own profile: "What do you know about me?"
- Making any personalized decision

WHEN TO SKIP:
- One-time situational info: "Book me a flight for Tuesday" (date, not a preference)
- Events that happened (use episodic memory for those, not semantic)
- Small talk, greetings, confirmations

WHEN FACTS CONFLICT:
- If the user explicitly corrects a preference, update immediately via learn_about_user
- If you detect a contradiction with stored knowledge, ASK the user before overwriting
- Newer explicit statements from the user override older ones

## Procedural Memory — Task Procedures
You have access to task-specific procedures via skills.

WHEN TO LOAD A PROCEDURE:
- User requests a booking → determine if domestic or international
- Call load_skill("domestic-booking") or load_skill("international-booking")
- Follow the loaded procedure step by step

FOR DETAILED DATA:
- Call read_skill_resource("domestic-booking", "budget-limits") for budget limits by level
- Call read_skill_resource("international-booking", "visa-checklist") for visa requirements

HOW TO CHOOSE:
- Same country (e.g. US to US) → domestic-booking
- Different country → international-booking
- If unsure, ask the user

ALWAYS follow the loaded procedure's checklist. Do not skip steps.

The current user is Sarah Chen (employee ID: E001, level: Senior, based in Seattle, USA).
Be concise and helpful.
"""

os.makedirs("prompts", exist_ok=True)
Path("prompts/v3.md").write_text(V3_INSTRUCTIONS, encoding="utf-8")
print(f"\nCreated: prompts/v3.md")
print(f"Size: {len(V3_INSTRUCTIONS)} characters, {len(V3_INSTRUCTIONS.splitlines())} lines")

SkillsProvider discovered:

  📋 domestic-booking: Step-by-step procedure for booking domestic (same-country) business travel. Use ...
     └── resource: <agent_framework._skills.SkillResource object at 0x000002DDC09FA3C0>

  📋 international-booking: Step-by-step procedure for booking international business travel. Use when emplo...
     └── resource: <agent_framework._skills.SkillResource object at 0x000002DDC2276D50>
     └── resource: <agent_framework._skills.SkillResource object at 0x000002DDC2275D10>

Tools exposed:
  🔧 load_skill
  🔧 read_skill_resource

Created: prompts/v3.md
Size: 2376 characters, 54 lines


C:\Users\divyesheth\AppData\Local\Temp\ipykernel_10356\2217907852.py:5: ExperimentalWarning: [SKILLS] SkillsProvider is experimental and may change or be removed in future versions without notice.
  procedures = SkillsProvider(skill_paths=["skills"])


In [ ]:
print("=== What changed: V2 → V3 ===\n")
show_diff("prompts/v2.md", "prompts/v3.md")

=== What changed: V2 → V3 ===

--- prompts/v2.md
+++ prompts/v3.md
@@ -31,18 +31,24 @@
 - If you detect a contradiction with stored knowledge, ASK the user before overwriting
 - Newer explicit statements from the user override older ones
 
-## Booking Procedure
-When the user requests a booking, follow these steps IN ORDER:
+## Procedural Memory — Task Procedures
+You have access to task-specific procedures via skills.
 
-1. **Check preferences** (semantic memory): query_user_knowledge for airline, hotel, seat preferences
-2. **Check history** (episodic memory): recall_events for past trips to the same destination
-3. **Verify budget**: Senior employees have a $800 flight limit, economy/economy-plus only.
-   - If the request exceeds the limit, explain the policy and suggest alternatives
-   - Do NOT silently book over-budget options
-4. **Explain your reasoning**: Tell the user WHY you're recommending what you recommend
-   - "Based on your preference for Delta and your past positive 

In [ ]:
async def demo_v3():
    instructions = load_instructions("prompts/v3.md")
    print(f"Loaded instructions from: prompts/v3.md ({len(instructions)} chars)\n")

    agent = Agent(
        client=FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        ),
        name="TravelAssistant",
        instructions=instructions,
        tools=[
            remember_event, recall_events,
            learn_about_user, query_user_knowledge,
        ],
        context_providers=[procedures],
    )
    turns = [
        "Hi, I'm Sarah Chen (E001). I need to book a trip to London next month for a client meeting.",
        "I've been to London before — you should have that in your records.",
    ]
    for user_msg in turns:
        print(f"User: {user_msg}")
        response = await agent.run(user_msg)
        print(f"Agent: {response.text}\n")

print("=== V3: + Task-Specific Procedures via SkillsProvider ===\n")
asyncio.run(demo_v3())

=== V3: + Task-Specific Procedures via SkillsProvider ===

Loaded instructions from: prompts/v3.md (2376 chars)

User: Hi, I'm Sarah Chen (E001). I need to book a trip to London next month for a client meeting.
Agent: Hi Sarah — London is an international trip, so I’ll follow our international booking process.

What I know so far:
- Airline preference: Delta
- Seat preference: aisle
- Hotel preference: Marriott
- Past London trip: Nov 20–25, 2025; stayed at Marriott Park Lane

A couple of required checks before booking:
- International travel requires travel insurance
- We need at least 14 days’ advance booking
- Do you have a valid UK visa already?

Also, you said “next month” — can you share your exact travel dates and whether you want me to book flights, hotel, or both?

User: I've been to London before — you should have that in your records.
Agent: Got it — I’ve noted that you’ve been to London before.



### Skills Orchestrate Memory

The agent loaded `international-booking`, then used the procedure to coordinate all memory systems:

```
Procedural memory (the conductor)
    ├── "Check preferences" → semantic memory
    ├── "Check history"     → episodic memory
    ├── "Verify budget"     → read_skill_resource (policy data)
    ├── "Travel insurance"  → apply policy rule
    └── "Check visa"        → episodic + ask user
```

**Key distinction**: Procedural memory is the *concept* (knowing how). `SkillsProvider` is the *delivery mechanism* — lazy-loaded SKILL.md files with bundled resources.


---
## When to Stop Refining

The feedback loop (V1 → V2 → V3) is powerful, but needs a stopping condition.

### Anti-Patterns
1. **Over-fitting** — Instructions so specific to one user's edge cases they break for others
2. **Oscillation** — V5 fixes V4, V6 re-breaks V5. Git log shows reverts
3. **Diminishing returns** — 80% → 95% took two iterations; 95% → 97% takes five
4. **Instruction bloat** — Three-page prompts cause the model to ignore sections

### Stopping Signals

| Signal | Rule of thumb |
|---|---|
| Eval scores plateau | Delta < meaningful threshold |
| Git shows reverts | You're undoing previous changes |
| Instruction length | Prompt exceeds ~1500 tokens |
| Edge case ratio | >50% of changes address <5% of conversations |

**Practical default — the 3-iteration rule:**
- **V1**: Baseline — get something working
- **V2**: Fix observed weaknesses — the biggest behavioral gaps
- **V3**: Polish — edge cases and refinements

Beyond V3, ask: "Do I need better *tools* or a different *approach*?" — not "What else can I add to the prompt?"


In [ ]:
# Show the evolution trail
print("=== Instruction Version History ===\n")

versions = sorted(Path("prompts").glob("v*.md"))
for v in versions:
    content = v.read_text(encoding="utf-8")
    lines = content.splitlines()
    sections = [l.strip("# ").strip() for l in lines if l.startswith("## ")]
    print(f"  {v.name:8s}  {len(lines):3d} lines  {len(content):5d} chars  Sections: {', '.join(sections)}")

print()
print("Skills directory:")
for p in sorted(Path("skills").rglob("*")):
    if p.is_file():
        print(f"  {p}")

print()
print("If you've initialized git in this directory, you can run:")
print('  git add prompts/ skills/')
print('  git commit -m "procedural memory: v1 → v2 → v3 with SkillsProvider"')
print('  git log --oneline prompts/')
print()
print("The git history IS the procedural memory — it records how the agent learned.")

=== Instruction Version History ===

  v1.md      35 lines   1644 chars  Sections: Episodic Memory — Past Events, Semantic Memory — Facts & Preferences
  v2.md      48 lines   2516 chars  Sections: Episodic Memory — Past Events, Semantic Memory — Facts & Preferences, Booking Procedure
  v3.md      54 lines   2376 chars  Sections: Episodic Memory — Past Events, Semantic Memory — Facts & Preferences, Procedural Memory — Task Procedures

Skills directory:
  skills\domestic-booking\budget-limits.json
  skills\domestic-booking\SKILL.md
  skills\international-booking\budget-limits.json
  skills\international-booking\SKILL.md
  skills\international-booking\visa-checklist.md

If you've initialized git in this directory, you can run:
  git add prompts/ skills/
  git commit -m "procedural memory: v1 → v2 → v3 with SkillsProvider"
  git log --oneline prompts/

The git history IS the procedural memory — it records how the agent learned.


---
## Putting It All Together

All four memory systems in one conversation: semantic update, international booking with full procedure, and a domestic trip for comparison.


In [ ]:
async def demo_full():
    instructions = load_instructions("prompts/v3.md")

    agent = Agent(
        client=FoundryChatClient(
            project_endpoint=PROJECT_ENDPOINT,
            model=MODEL_DEPLOYMENT,
            credential=credential,
        ),
        name="TravelAssistant",
        instructions=instructions,
        tools=[
            remember_event, recall_events,
            learn_about_user, query_user_knowledge,
        ],
        context_providers=[procedures],
    )
    turns = [
        # Semantic: update a preference
        "I've switched from Delta to United — their international routes are better.",
        # Procedural + episodic + semantic: full booking
        "Plan a trip to London next month for a client meeting.",
        # Domestic trip for comparison
        "Also, I need a quick trip to NYC next week for a conference.",
    ]
    for user_msg in turns:
        print(f"User: {user_msg}")
        response = await agent.run(user_msg)
        print(f"Agent: {response.text}\n")
        print("-" * 60 + "\n")

print("=== Full Demo: All Memory Types + SkillsProvider ===\n")
asyncio.run(demo_full())

# Show state
print("\n--- Memory State After Demo ---")
print(f"\nEpisodic events: {len(_episodic_store)}")
for e in _episodic_store[-3:]:
    print(f"  [{e['timestamp'][:10]}] {e['description'][:80]}")

print(f"\nSemantic facts: {len(_knowledge_graph.all_facts('E001'))}")
for f in _knowledge_graph.all_facts("E001"):
    print(f"  ({f['subject']}, {f['relationship']}, {f['object']}/{f['object_type']})")

print(f"\nSkills: {list(procedures._skills.keys())}")

=== Full Demo: All Memory Types + SkillsProvider ===

User: I've switched from Delta to United — their international routes are better.
Agent: Got it — I’ll prefer United for your flights going forward.

------------------------------------------------------------

User: Plan a trip to London next month for a client meeting.
Agent: London is an international trip, so I’ll follow international booking steps.

What I know about your preferences:
- Airline: United
- Hotel: Marriott
- Seat: aisle
- Dietary: vegetarian

I also see you’ve been to London before, including a prior client-meeting trip.

Before I build the plan, I need two quick things:
1. Your exact travel dates next month
2. Whether you have a valid UK visa already

Also note:
- International trips require travel insurance
- Company policy requires booking at least 14 days in advance

Send me the dates and visa status, and I’ll put together a recommended itinerary.

------------------------------------------------------------


### What Just Happened

The agent used **all four memory systems** in one conversation:

| Action | Memory Type | Tool/File |
|---|---|---|
| "Switched to United" | **Semantic** — updated preference | `learn_about_user` |
| "Plan London trip" | **Procedural** — loaded int'l procedure | `load_skill` |
| Checked past London trip | **Episodic** — recalled history | `recall_events` |
| Applied travel insurance rule | **Procedural** — followed checklist | Skill resource |
| Checked budget limits | **Procedural** — queried budget data | `read_skill_resource` |
| Checked preferences before booking | **Semantic** — queried knowledge | `query_user_knowledge` |
| "Quick trip to NYC" | **Procedural** — loaded domestic procedure | `load_skill` |
| Referenced past NYC experience | **Episodic** — recalled conference trip | `recall_events` |

**Procedural memory orchestrates everything.** The skill files tell the agent which other memory systems to consult, in what order, and what to do with the results.

---
## Summary

### Four Memory Types

| | Session (Module 1) | Episodic (Module 2) | Semantic (Module 3) | Procedural (Module 4) |
|---|---|---|---|---|
| **Stores** | Current conversation | Past events | Facts & preferences | Behaviors & procedures |
| **Answers** | "What did we just discuss?" | "What happened?" | "What is true?" | "How should I act?" |
| **Structure** | Message list | Event records | Knowledge graph | SKILL.md files |
| **Update pattern** | Automatic | `@tool` | `@tool` | Human refinement |
| **Version control** | N/A | Cosmos DB | Neo4j | SKILL.md files + git |
| **Key API** | `AgentSession` | `remember_event` | `learn_about_user` | `SkillsProvider` (`load_skill`, `read_skill_resource`) |

### Key Insights

1. **Instructions are memory.** The agent's behavior comes from its instructions — not from code. Treating them as versioned files makes them reviewable, diffable, and auditable.

2. **Procedures orchestrate other memories.** The booking procedure says "check preferences (semantic), check history (episodic), verify budget." It's the playbook that ties everything together.

3. **The feedback loop has a stopping condition.** Observe → refine → verify is powerful, but watch for over-fitting, oscillation, bloat, and diminishing returns. Three iterations is a good default.

4. **Task-specific procedures reduce complexity.** Instead of one giant instruction set, load the right playbook for the right task. International trips get visa checks; domestic trips don't.

5. **SkillsProvider is the delivery mechanism, not the memory.** Procedural memory is the concept (knowing *how* to do things). `SkillsProvider` is how the agent framework delivers it — lazy-loaded SKILL.md files with bundled resources. The same pattern can deliver any knowledge type.

### More Than One Way

`SkillsProvider` + git is one approach to delivering procedural memory — framework-native, with progressive disclosure and bundled resources. But procedural memory itself can be implemented many ways:

- **Skill libraries** (Voyager) — agent builds reusable executable procedures from successful tasks
- **Self-editing prompts** (MemGPT/Letta) — agent modifies its own instructions at runtime
- **Reflection-based learning** (Reflexion) — agent writes post-task observations that feed back into future context
- **Prompt optimization pipelines** — automated eval → optimizer → human review cycles

The same is true for episodic and semantic memory — each has multiple valid implementation strategies with different tradeoffs. We'll compare these approaches across all three memory types in a later module.

---
## What's Next

We've built four memory systems, but each module taught them **in isolation**. The agent in this module had all four memory types — but we set up each one from scratch.

In **Module 5: Combined Memory**, we'll bring all four memory systems together with shared infrastructure — a single agent configuration that leverages episodic, semantic, and procedural memory with persistent backends. No more rebuilding tools in each notebook.

In [16]:
# Optional: Clean up files created by this module
# Uncomment and run to remove generated prompt and skill files

# import shutil
# if Path("prompts").exists():
#     shutil.rmtree("prompts")
#     print("Removed prompts/")
# if Path("skills").exists():
#     shutil.rmtree("skills")
#     print("Removed skills/")